In [ ]:
import sys
sys.path.append('..')

import shap
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # prevents display errors on Windows
import matplotlib.pyplot as plt

from src.preprocess import load_and_clean, NUM_COLS, CAT_COLS

# Load trained pipeline
pipeline = joblib.load('../models/churn_pipeline.pkl')
print('Pipeline loaded')
print('Steps:', [s[0] for s in pipeline.steps])

# Load and prep data
df = load_and_clean('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
FEATURE_COLS = NUM_COLS + CAT_COLS
X = df[FEATURE_COLS]
y = df['Churn']

# Use 500 rows for SHAP — faster, still representative
X_sample = X.sample(500, random_state=42).reset_index(drop=True)
y_sample = y.loc[X_sample.index].reset_index(drop=True)

print(f'Sample shape: {X_sample.shape}')
print('Ready for SHAP analysis')

In [ ]:
# Transform data through the preprocessor step only
preprocessor = pipeline.named_steps['preprocessor']
X_transformed = preprocessor.transform(X_sample)

# Get the XGBoost model from the pipeline
xgb_model = pipeline.named_steps['model']

# Create SHAP explainer — TreeExplainer is fast for XGBoost
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_transformed)

print('SHAP values computed')
print(f'Shape: {shap_values.shape}')
print(f'Base value (avg prediction): {explainer.expected_value:.4f}')

In [ ]:
feature_names = NUM_COLS + CAT_COLS

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_transformed,
    feature_names=feature_names,
    show=False,
    plot_size=None
)
plt.title('SHAP Feature Importance — Customer Churn', 
          fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../reports/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> reports/shap_summary.png')

In [ ]:
# Find a high-risk customer to explain (predicted churn prob > 0.7)
probas = pipeline.predict_proba(X_sample)[:, 1]
high_risk_idx = np.where(probas > 0.7)[0]

if len(high_risk_idx) == 0:
    high_risk_idx = [np.argmax(probas)]  # fallback: highest prob

idx = high_risk_idx[0]
print(f'Explaining customer index {idx}')
print(f'Churn probability: {probas[idx]:.1%}')
print(f'Actual label: {"Churned" if y_sample.iloc[idx] == 1 else "Did not churn"}')
print(f'Contract: {X_sample.iloc[idx]["Contract"]}')
print(f'Tenure: {X_sample.iloc[idx]["tenure"]} months')
print(f'Monthly charges: ${X_sample.iloc[idx]["MonthlyCharges"]}')

# Build SHAP Explanation object
explanation = shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=X_transformed[idx],
    feature_names=feature_names
)

# Waterfall plot
plt.figure(figsize=(10, 7))
shap.waterfall_plot(explanation, show=False, max_display=12)
plt.title(f'Why is this customer high risk? (Churn prob: {probas[idx]:.1%})',
          fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('../reports/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> reports/shap_waterfall.png')

In [ ]:
## Key findings from SHAP analysis

### Global feature importance
- **Contract type** is the strongest churn driver — month-to-month 
  customers have large positive SHAP values pushing toward churn
- **Tenure** is strongly negative for churn — longer customers are 
  much less likely to leave
- **Monthly charges** above ~$70 push toward churn, especially 
  combined with fiber optic internet

### Per-customer explanation
The waterfall plot shows how each feature moves the prediction 
from the base rate (26%) to the final probability for one customer.
Red bars push toward churn, blue bars push away.

### Business recommendation
Target retention efforts at customers who are:
1. On month-to-month contracts (switch offer)
2. Tenure under 6 months (onboarding improvement)
3. Paying over $70/month with no tech support (bundle offer)